# Lullaby — Multi-Night Comparison

Compare sleep data across multiple nights to assess:
- Consistency of REM-discriminating features
- Night-over-night trends
- Data quality improvements

**Usage:** Place exported session JSONs in `../data/`, or use mock data for testing.

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from lullaby.loader import load_all_sessions, generate_mock_session
from lullaby.features import align_to_epochs, get_feature_columns
from lullaby.visualization import plot_night_overview, STAGE_COLORS, STAGE_LABELS, STAGE_ORDER
from lullaby.statistics import compare_rem_vs_non_rem, format_statistics_table
from lullaby.quality import assess_quality

print('Lullaby analysis toolkit loaded.')

## 1. Load Sessions

Load real sessions from `../data/`, or generate mock sessions for testing.

In [ ]:
USE_MOCK = True  # Set to False when you have real data in ../data/

if USE_MOCK:
    sessions = [
        generate_mock_session(duration_hours=7.5, seed=42),
        generate_mock_session(duration_hours=8.0, seed=123),
        generate_mock_session(duration_hours=6.5, seed=789),
    ]
    print(f'Generated {len(sessions)} mock sessions')
else:
    sessions = load_all_sessions('../data/')
    print(f'Loaded {len(sessions)} sessions from ../data/')

for i, s in enumerate(sessions):
    print(f'  Night {i+1}: {s.duration_hours:.1f}h, '
          f'{len(s.heart_rate):,} HR, {len(s.sleep_stages)} stages')

## 2. Per-Night Quality Summary

In [ ]:
quality_rows = []
for i, s in enumerate(sessions):
    q = assess_quality(s)
    quality_rows.append({
        'night': i + 1,
        'duration_h': q['overview']['duration_hours'],
        'hr_coverage_%': q['heart_rate']['coverage_pct'],
        'accel_coverage_%': q['accelerometer']['coverage_pct'],
        'sonar_coverage_%': q['sonar']['coverage_pct'],
        'stage_coverage_%': q['sleep_stages']['coverage_pct'],
        'packets_received': q['connectivity']['packets_received'],
        'phone_drain_%': q['battery'].get('phone_drain_pct', None),
        'watch_drain_%': q['battery'].get('watch_drain_pct', None),
    })

quality_df = pd.DataFrame(quality_rows)
quality_df

## 3. Night Overview Gallery

Side-by-side overview plots for each night.

In [ ]:
for i, s in enumerate(sessions):
    print(f'\n--- Night {i+1} ({s.duration_hours:.1f}h) ---')
    fig = plot_night_overview(s, figsize=(14, 10))
    plt.show()
    plt.close(fig)

## 4. Epoch Alignment for All Nights

In [ ]:
all_epochs = []
for i, s in enumerate(sessions):
    ep = align_to_epochs(s)
    ep['night'] = i + 1
    all_epochs.append(ep)
    print(f'Night {i+1}: {len(ep)} epochs')

combined = pd.concat(all_epochs, ignore_index=True)
print(f'\nCombined: {len(combined)} epochs across {len(sessions)} nights')
print(f'Sleep stage distribution (combined):')
print(combined['sleep_stage'].value_counts())

## 5. Aggregate Feature Distributions

Feature distributions across ALL nights combined.

In [ ]:
features = get_feature_columns(combined)
valid = combined[combined['sleep_stage'].isin(STAGE_ORDER)].copy()

ncols = 3
nrows = -(-len(features) // ncols)  # Ceiling division
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3 * nrows))
axes = axes.flatten()

palette = {s: STAGE_COLORS[s] for s in STAGE_ORDER}

for idx, feat in enumerate(features):
    ax = axes[idx]
    data = valid[['sleep_stage', feat]].dropna()
    if data.empty:
        ax.set_visible(False)
        continue
    sns.boxplot(
        data=data, x='sleep_stage', y=feat,
        order=[s for s in STAGE_ORDER if s in data['sleep_stage'].values],
        palette=palette, ax=ax, linewidth=0.8,
    )
    ax.set_title(f'{feat} (all nights)', fontsize=10)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)

for idx in range(len(features), len(axes)):
    axes[idx].set_visible(False)

fig.suptitle('Feature Distributions by Sleep Stage (All Nights Combined)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Per-Night REM vs Non-REM Statistics

In [ ]:
for i, ep in enumerate(all_epochs):
    print(f'\n{"=" * 60}')
    print(f'  Night {i+1}')
    print(f'{"=" * 60}')
    stats_df = compare_rem_vs_non_rem(ep)
    print(format_statistics_table(stats_df))

## 7. Combined REM vs Non-REM Statistics

More statistical power by pooling all nights together.

In [ ]:
print('Combined statistics across all nights:')
combined_stats = compare_rem_vs_non_rem(combined)
print(format_statistics_table(combined_stats))

print(f'\n\nSignificant features (combined, Bonferroni-corrected):')
sig = combined_stats[combined_stats['significant']]
for _, row in sig.iterrows():
    print(f"  {row['feature']}: p={row['p_corrected']:.2e}, d={row['cohens_d']:.2f}")

## 8. Feature Consistency Across Nights

Do the same features discriminate REM across all nights?

In [ ]:
# Collect effect sizes per night per feature
consistency_rows = []
for i, ep in enumerate(all_epochs):
    stats_df = compare_rem_vs_non_rem(ep)
    for _, row in stats_df.iterrows():
        consistency_rows.append({
            'night': i + 1,
            'feature': row['feature'],
            'cohens_d': abs(row['cohens_d']),
            'significant': row['significant'],
        })

consistency = pd.DataFrame(consistency_rows)

if not consistency.empty:
    # Pivot: features as rows, nights as columns, values = Cohen's d
    pivot = consistency.pivot(index='feature', columns='night', values='cohens_d')
    pivot['mean_d'] = pivot.mean(axis=1)
    pivot['std_d'] = pivot.std(axis=1)
    pivot = pivot.sort_values('mean_d', ascending=False)
    
    print('Feature consistency (|Cohen d| across nights):')
    print(pivot.round(3).to_string())
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    pivot_plot = pivot.drop(columns=['mean_d', 'std_d'])
    pivot_plot.plot(kind='bar', ax=ax, width=0.8)
    ax.set_ylabel('|Cohen d|')
    ax.set_title('REM Discrimination Consistency Across Nights')
    ax.axhline(y=0.8, color='red', linestyle='--', alpha=0.5, label='Large effect')
    ax.axhline(y=0.5, color='orange', linestyle='--', alpha=0.5, label='Medium effect')
    ax.legend()
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No consistency data available.')

## 9. Summary

Which features are the most reliable REM discriminators across multiple nights?

In [ ]:
print('=' * 50)
print('  MULTI-NIGHT SUMMARY')
print('=' * 50)
print(f'\nNights analyzed: {len(sessions)}')
print(f'Total epochs: {len(combined)}')

if not consistency.empty:
    # Features that are significant in ALL nights
    sig_counts = consistency.groupby('feature')['significant'].sum()
    always_sig = sig_counts[sig_counts == len(sessions)].index.tolist()
    
    print(f'\nFeatures significant in ALL {len(sessions)} nights:')
    if always_sig:
        for f in always_sig:
            mean_d = consistency[consistency['feature'] == f]['cohens_d'].mean()
            print(f'  {f}: mean |d| = {mean_d:.2f}')
    else:
        print('  None — may need more data or better signal quality')
    
    # Best overall features by mean effect size
    mean_effects = consistency.groupby('feature')['cohens_d'].mean().sort_values(ascending=False)
    print(f'\nTop 5 features by mean effect size across nights:')
    for feat, d in mean_effects.head(5).items():
        print(f'  {feat}: mean |d| = {d:.3f}')
    
    print(f'\nRecommendation for Phase 2 model features:')
    recommended = mean_effects[mean_effects > 0.3].index.tolist()
    if recommended:
        print(f'  Include: {", ".join(recommended)}')
    else:
        print('  Need more nights of data to identify reliable features')